# Шаг 6. Связь с финальным проектом: делаем маркировку трубы читаемой

В финальном проекте нужно **прочитать маркировку** на трубе. Распознавателю текста (OCR) мешают
ржавчина, блики, неровное освещение и то, что надпись перевёрнута. Поэтому перед распознаванием
картинку готовят **цепочкой (конвейером) операций** — каждая решает одну проблему.

**Конвейер:** вырезать → увеличить → повернуть → серый → контраст (CLAHE) → сгладить → адаптивный порог.

**Структура ноутбука (алгоритм):**
1. Импортируем OpenCV, NumPy и помощники, загружаем полное фото трубы.
2. Шаг за шагом применяем семь операций, сохраняя результат каждой в отдельную переменную.
3. Склеиваем промежуточные результаты **вертикально** в одну ленту, чтобы видеть эффект каждого шага.
4. Сохраняем ленту в `marking_steps.jpg` и сравниваем, на каком шаге текст читается лучше.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session02")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())

## 1. Импорт библиотек и общих помощников

- `cv2` — библиотека **OpenCV**: чтение, обработка и показ изображений.
- `numpy` (`np`) — работа с массивами чисел. Картинка в OpenCV — это и есть массив NumPy.
- `common` — наш модуль из этой же папки (`common.py`) с маленькими помощниками:

| Имя | Что делает |
|-----|------------|
| `PIPE` | путь к фотографии трубы из датасета (`dataset/images/train/...`) |
| `load(path)` | прочитать картинку (работает и с русскими буквами в пути) |
| `save(path, img)` | сохранить картинку в файл |
| `show_nb(img)` | показать картинку прямо в ноутбуке, под ячейкой |
| `label(img, text)` | подписать картинку чёрной полосой сверху (только латиница) |
| `grid(images, cols)` | склеить несколько картинок одинакового размера в таблицу |
| `fit(img, w, h)` | уменьшить картинку, если она не влезает в заданный размер |

> Ноутбук нужно запускать из папки `session02`, иначе `import common` не найдёт модуль.

In [ ]:
import cv2
import numpy as np

from common import PIPE, label, load, save, show_nb

img = load(PIPE)

## 2. Шаги 1–3: вырезать, увеличить, повернуть

1. **Вырезать** — срез `img[90:240, 560:1260]` оставляет только полосу с маркировкой (как в шаге 1).
   Лишний фон только мешает и замедляет обработку.
2. **Увеличить в 2 раза** — `cv2.resize(crop, None, fx=2, fy=2, ...)`: вместо точного размера (`None`) задаём коэффициенты.
   `INTER_CUBIC` (бикубическая интерполяция) считает новые пиксели по 4 × 4 соседям — буквы получаются глаже, чем при простом
   копировании пикселей. Распознаватели текста лучше работают с крупными буквами.
3. **Повернуть на 180°** — на этом фото надпись вверх ногами.

In [ ]:
crop = img[90:240, 560:1260]                                  # 1. вырезать область маркировки
big = cv2.resize(crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)  # 2. увеличить в 2 раза
rotated = cv2.rotate(big, cv2.ROTATE_180)                     # 3. текст был вверх ногами
show_nb(rotated)

## 3. Шаги 4–5: серый и контраст CLAHE

4. **Серый** — цвет для чтения текста не нужен, а один канал обрабатывается втрое быстрее.
5. **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) — «умное» повышение контраста:
   - картинка делится на сетку 8 × 8 кусочков (`tileGridSize`);
   - в каждом кусочке контраст растягивается **отдельно**, по его собственной гистограмме яркостей;
   - `clipLimit=3.0` ограничивает усиление, чтобы не раздуть шум на однородных участках;
   - на стыках кусочков результаты плавно смешиваются.

   Обычное выравнивание по всей картинке сразу «ослепло» бы от бликов: яркое пятно в одном месте
   определяло бы контраст везде. CLAHE же вытягивает тёмные места и не трогает пересвеченные.

`createCLAHE` создаёт объект-обработчик, а `.apply(gray)` применяет его к картинке.

In [ ]:
gray = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)              # 4. цвет не нужен
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))   # 5. контраст по кусочкам:
contrast = clahe.apply(gray)                                  #    блики не мешают тёмным местам
show_nb(contrast)

## 4. Шаги 6–7: сгладить и адаптивный порог

6. **Медианный фильтр 3 × 3** убирает мелкие точки ржавчины (вспомните шаг 3: медиана лучше всего справляется с одиночными точками),
   почти не размывая края букв.
7. **Адаптивный порог** превращает картинку в чёрно-белую, но порог у каждого пикселя **свой**:
   - `ADAPTIVE_THRESH_GAUSSIAN_C` — порог = взвешенное (по Гауссу) среднее яркости в окрестности `31 × 31` минус `C`;
   - `C = -5` — отрицательное, значит порог чуть **выше** среднего: белыми станут только пиксели заметно ярче окружения — светлые буквы;
   - размер окна `31` (нечётный) должен быть больше толщины букв, иначе внутренность буквы сочтётся «фоном».

   Глобальный порог (одно число на всю картинку, как в шаге 2) здесь бы не сработал: освещение по трубе неравномерное.

In [ ]:
smooth = cv2.medianBlur(contrast, 3)                          # 6. убрать мелкие точки ржавчины
binary = cv2.adaptiveThreshold(smooth, 255,                   # 7. свой порог для каждого участка
                               cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, -5)
show_nb(binary)

## 5. Лента всех шагов

`np.vstack` ставит картинки друг под другом (все они одной ширины — 1400 пикселей после увеличения).
Серые картинки `label` превращает в трёхканальные, поэтому их можно склеивать с цветными.

In [ ]:
steps = np.vstack([
    label(big, "1-2. crop + resize x2"),
    label(rotated, "3. rotate 180"),
    label(gray, "4. grayscale"),
    label(contrast, "5. CLAHE contrast"),
    label(binary, "6-7. median + adaptive threshold"),
])

save("marking_steps.jpg", steps)
show_nb(steps, 1400, 1600)
print("Вопрос: на каком шаге текст читается лучше всего? Почему не на последнем?")

## Попробуйте сами

1. Меняйте `clipLimit` (1, 3, 8) и `tileGridSize` ((4, 4), (16, 16)). Как меняется вид маркировки?
2. Поменяйте `C` в адаптивном пороге на `+5`. Почему картинка «вывернулась»?
3. Попробуйте размер окна порога `15` и `61`. Что происходит с буквами?
4. Возьмите другое фото из `dataset/images/train` — подойдёт ли тот же конвейер без изменений? Что придётся подбирать заново?